In [0]:
# Create Silver schema

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

print("Silver schema created successfully!")

Silver schema created successfully!


In [0]:
spark.sql("""
SHOW SCHEMAS IN workspace
""").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|information_schema|
|silver            |
|students_intro    |
+------------------+



In [0]:
from pyspark.sql import functions as F

# Read Bronze Delta tables

bronze_tables = {
    "customers": "workspace.bronze.customers",
    "geolocation": "workspace.bronze.geolocation",
    "order_items": "workspace.bronze.order_items",
    "payments": "workspace.bronze.payments",
    "reviews": "workspace.bronze.reviews",
    "orders": "workspace.bronze.orders",
    "products": "workspace.bronze.products",
    "sellers": "workspace.bronze.sellers",
    "category_translation": "workspace.bronze.category_translation"
}

# Load tables into DataFrames

datasets = {}

for name, table_name in bronze_tables.items():

    datasets[name] = spark.table(table_name)

    print(f"Loaded: {name}")

print("\nTotal Bronze tables loaded:", len(datasets))

Loaded: customers
Loaded: geolocation
Loaded: order_items
Loaded: payments
Loaded: reviews
Loaded: orders
Loaded: products
Loaded: sellers
Loaded: category_translation

Total Bronze tables loaded: 9


In [0]:
# Inspect schemas of all nine datasets

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(f"DATASET: {name.upper()}")
    print("=" * 60)

    df.printSchema()


DATASET: CUSTOMERS
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


DATASET: GEOLOCATION
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


DATASET: ORDER_ITEMS
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price

In [0]:

# Profile NULL values across all datasets

for name, df in datasets.items():

    print(f"\nNULL PROFILE: {name.upper()}")

    null_expr = [
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in df.columns
    ]

    null_profile = df.agg(*null_expr)

    display(null_profile)



NULL PROFILE: CUSTOMERS


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,source_file
0,0,0,0,0,0,0



NULL PROFILE: GEOLOCATION


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,ingestion_timestamp,source_file
0,0,0,0,0,0,0



NULL PROFILE: ORDER_ITEMS


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,source_file
0,0,0,0,0,0,0,0,0


... (8 outputs left out)


NULL PROFILE: SELLERS


seller_id,seller_zip_code_prefix,seller_city,seller_state,ingestion_timestamp,source_file
0,0,0,0,0,0



NULL PROFILE: CATEGORY_TRANSLATION


product_category_name,product_category_name_english,ingestion_timestamp,source_file
0,0,0,0


In [0]:
# Check exact duplicate records across all datasets

for name, df in datasets.items():

    total_records = df.count()

    unique_records = df.dropDuplicates().count()

    duplicate_records = total_records - unique_records

    print(f"""
    Dataset: {name}
    Total Records: {total_records}
    Unique Records: {unique_records}
    Duplicate Records: {duplicate_records}
    """)


    Dataset: customers
    Total Records: 99441
    Unique Records: 99441
    Duplicate Records: 0
    

    Dataset: geolocation
    Total Records: 1000163
    Unique Records: 738332
    Duplicate Records: 261831
    

    Dataset: order_items
    Total Records: 112650
    Unique Records: 112650
    Duplicate Records: 0
    

    Dataset: payments
    Total Records: 103886
    Unique Records: 103886
    Duplicate Records: 0
    

    Dataset: reviews
    Total Records: 104162
    Unique Records: 104077
    Duplicate Records: 85
    

    Dataset: orders
    Total Records: 99441
    Unique Records: 99441
    Duplicate Records: 0
    

    Dataset: products
    Total Records: 32951
    Unique Records: 32951
    Duplicate Records: 0
    

    Dataset: sellers
    Total Records: 3095
    Unique Records: 3095
    Duplicate Records: 0
    

    Dataset: category_translation
    Total Records: 71
    Unique Records: 71
    Duplicate Records: 0
    


In [0]:
display(
    datasets["orders"]
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

order_status,count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
display(
    datasets["orders"]
    .filter(
        F.col("order_delivered_customer_date").isNull()
    )
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

order_status,count
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


In [0]:
# Define primary/business keys for validation

key_columns = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id"],
    "category_translation": ["product_category_name"]
}

# Check NULL and duplicate keys

for name, columns in key_columns.items():

    df = datasets[name]

    print(f"\nKEY VALIDATION: {name.upper()}")

    # Count NULL keys
    null_condition = None

    for column in columns:
        condition = F.col(column).isNull()

        null_condition = (
            condition if null_condition is None
            else null_condition | condition
        )

    null_count = df.filter(null_condition).count()

    # Count duplicate key combinations
    duplicate_count = (
        df.groupBy(columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("Key columns:", columns)
    print("NULL key records:", null_count)
    print("Duplicate key combinations:", duplicate_count)


KEY VALIDATION: CUSTOMERS
Key columns: ['customer_id']
NULL key records: 0
Duplicate key combinations: 0

KEY VALIDATION: ORDERS
Key columns: ['order_id']
NULL key records: 0
Duplicate key combinations: 0

KEY VALIDATION: PRODUCTS
Key columns: ['product_id']
NULL key records: 0
Duplicate key combinations: 0

KEY VALIDATION: SELLERS
Key columns: ['seller_id']
NULL key records: 0
Duplicate key combinations: 0

KEY VALIDATION: ORDER_ITEMS
Key columns: ['order_id', 'order_item_id']
NULL key records: 0
Duplicate key combinations: 0

KEY VALIDATION: PAYMENTS
Key columns: ['order_id', 'payment_sequential']
NULL key records: 0
Duplicate key combinations: 0

KEY VALIDATION: REVIEWS
Key columns: ['review_id']
NULL key records: 1
Duplicate key combinations: 955

KEY VALIDATION: CATEGORY_TRANSLATION
Key columns: ['product_category_name']
NULL key records: 0
Duplicate key combinations: 0


In [0]:
# Identify duplicated review IDs

reviews_df = datasets["reviews"]

duplicate_review_ids = (
    reviews_df
    .filter(F.col("review_id").isNotNull())
    .groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Number of duplicated review IDs:",
    duplicate_review_ids.count()
)

display(
    duplicate_review_ids
    .orderBy(F.desc("count"))
    .limit(20)
)

Number of duplicated review IDs: 955


review_id,count
"""",29
"Recomendo""",21
"Recomendo.""",21
"Obrigada""",18
"Recomendo!""",14
"Recomendo """,14
"Obrigado""",9
"Parabéns """,9
"Obrigada!""",8
"Recomendo. """,8


In [0]:
# Retrieve complete records for duplicate review IDs

duplicate_review_records = reviews_df.join(
    duplicate_review_ids.select("review_id"),
    on="review_id",
    how="inner"
)

display(
    duplicate_review_records
    .orderBy("review_id")
    .limit(50)
)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
"""",2017-09-23 00:00:00,2017-09-23 22:54:13,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2017-10-18 00:00:00,2017-10-29 21:44:21,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2018-05-17 00:00:00,2018-05-26 16:45:33,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2018-03-29 00:00:00,2018-04-02 21:15:58,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2018-04-19 00:00:00,2018-04-20 00:44:33,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2017-09-29 00:00:00,2017-10-01 18:26:00,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2017-03-16 00:00:00,2017-03-16 21:16:12,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2017-05-17 00:00:00,2017-05-23 11:32:01,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2018-04-15 00:00:00,2018-04-17 21:51:39,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv
"""",2017-10-25 00:00:00,2017-10-30 21:29:54,null,null,null,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv


In [0]:
# Inspect records with missing review IDs

display(
    reviews_df.filter(
        F.col("review_id").isNull()
    )
)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
null,material de boa qualidade.Agora é amarelo mesmo,pensei que era mais fosco,"mas e muito bonito.eu recomendo""",2018-01-06 00:00:00,2018-01-08 14:20:31,null,2026-11-14T17:50:18.399Z,dbfs:/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv


In [0]:
# Review data quality investigation

reviews_df = datasets["reviews"]

# Exclude ingestion metadata from business comparisons
business_columns = [
    c for c in reviews_df.columns
    if c not in ["ingestion_timestamp", "source_file"]
]

# Remove exact business-record duplicates for analysis
reviews_deduplicated = reviews_df.dropDuplicates(business_columns)

# Identify review IDs with conflicting records
review_id_analysis = (
    reviews_deduplicated
    .filter(F.col("review_id").isNotNull())
    .groupBy("review_id")
    .agg(
        F.count("*").alias("record_count")
    )
)

conflicting_review_ids = (
    review_id_analysis
    .filter(F.col("record_count") > 1)
)

print("Duplicate review ID groups:",
      conflicting_review_ids.count())

print("NULL review IDs:",
      reviews_df.filter(
          F.col("review_id").isNull()
      ).count())

display(
    conflicting_review_ids
    .orderBy(F.desc("record_count"))
    .limit(10)
)

Duplicate review ID groups: 892
NULL review IDs: 1


review_id,record_count
"""",29
"Recomendo.""",21
"Recomendo""",21
"Obrigada""",17
"Recomendo """,14
"Recomendo!""",14
"Parabéns """,9
"Obrigado""",9
"Recomendo. """,8
"Obrigada!""",8


In [0]:
# Inspect the complete schema
reviews_df.printSchema()

# Display actual records with all business columns
display(
    reviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    ).limit(10)
)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:
# Display review_id alongside the other identifying fields
reviews_df.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message"
).show(20, truncate=False)

# Inspect suspicious review IDs
display(
    reviews_df
    .filter(
        F.col("review_id").isNull() |
        (~F.col("review_id").rlike("^[0-9a-fA-F-]{32,36}$"))
    )
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    )
    .limit(20)
)

+------------------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_id                                 |order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                        |
+------------------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|7bc2406110b926393aa56f80a40eba40          |73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                        

review_id,order_id,review_score,review_comment_title,review_comment_message
",2018-02-16 00:00:00,2018-02-20 10:52:22",null,null,null,null
A entrega foi efetuada muito antes do prazo dado.,null,null,null,null
O produto já começou a ser usado e até o presente,null,null,null,null
"sem problemas.""",2018-03-23 00:00:00,2018-04-01 00:27:51,null,null
"Faltou Controle Remoto para Ar Condicionado Consul""",2018-01-28 00:00:00,2018-02-04 11:06:15,null,null
Estou satisfeita,null,null,null,null
",2017-06-02 00:00:00,2017-06-03 13:12:07",null,null,null,null
",2017-08-09 00:00:00,2017-08-09 12:03:45",null,null,null,null
",2017-11-22 00:00:00,2017-11-22 14:45:40",null,null,null,null
"Muito obrigada.... """,2017-06-24 00:00:00,2017-06-29 13:48:21,null,null


In [0]:
from pyspark.sql import functions as F

reviews_path = "/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv"

# Read CSV with multiline and quote handling
corrected_reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(reviews_path)
)

# Add ingestion metadata
corrected_reviews = (
    corrected_reviews
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("olist_order_reviews_dataset.csv"))
)

print("Corrected review count:", corrected_reviews.count())

display(corrected_reviews.limit(10))

Corrected review count: 99224


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47,2026-11-15T22:17:45.901Z,olist_order_reviews_dataset.csv


In [0]:
(
    corrected_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.reviews")
)

print("Bronze reviews table successfully updated!")
print(
    "Updated record count:",
    spark.table("workspace.bronze.reviews").count()
)

Bronze reviews table successfully updated!
Updated record count: 99224


In [0]:
datasets["reviews"] = spark.table("workspace.bronze.reviews")

In [0]:
reviews_df = datasets["reviews"]

# Remove ingestion metadata for business-level checks
review_columns = [
    c for c in reviews_df.columns
    if c not in ["ingestion_timestamp", "source_file"]
]

exact_duplicates = (
    reviews_df.count()
    - reviews_df.dropDuplicates(review_columns).count()
)

null_review_ids = reviews_df.filter(
    F.col("review_id").isNull()
).count()

duplicate_review_ids = (
    reviews_df
    .filter(F.col("review_id").isNotNull())
    .groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total reviews:", reviews_df.count())
print("Exact duplicate records:", exact_duplicates)
print("NULL review IDs:", null_review_ids)
print("Duplicate review ID groups:", duplicate_review_ids)

Total reviews: 99224
Exact duplicate records: 0
NULL review IDs: 0
Duplicate review ID groups: 789


In [0]:
# Analyze repeated review IDs
review_id_details = (
    reviews_df
    .groupBy("review_id")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.countDistinct("review_score").alias("distinct_scores"),
        F.min("review_creation_date").alias("first_review_date"),
        F.max("review_creation_date").alias("last_review_date")
    )
    .filter(F.col("record_count") > 1)
    .orderBy(F.desc("record_count"))
)

print("Repeated review ID groups:", review_id_details.count())

display(review_id_details.limit(15))

Repeated review ID groups: 789


review_id,record_count,distinct_orders,distinct_scores,first_review_date,last_review_date
ddc52555ca27b0fe67d5255147682d2d,3,3,1,2017-08-11 00:00:00,2017-08-11 00:00:00
44e9f871226d8a130de3fc39dfbdf0c5,3,3,1,2018-02-22 00:00:00,2018-02-22 00:00:00
2d6ac45f859465b5c185274a1c929637,3,3,1,2018-04-07 00:00:00,2018-04-07 00:00:00
e44840754f12fad2b8646712121b349a,3,3,1,2017-11-01 00:00:00,2017-11-01 00:00:00
08528f70f579f0c830189efc523d2182,3,3,1,2018-08-03 00:00:00,2018-08-03 00:00:00
0c76e7a547a531e7bf9f0b99cba071c1,3,3,1,2017-08-31 00:00:00,2017-08-31 00:00:00
c444278834184f72b1484dfe47de7f97,3,3,1,2017-02-08 00:00:00,2017-02-08 00:00:00
308316408775d1600dad81bd3184556d,3,3,1,2017-09-07 00:00:00,2017-09-07 00:00:00
3415c9f764e478409e8e0660ae816dd2,3,3,1,2018-07-31 00:00:00,2018-07-31 00:00:00
4219a80ab469e3fc9901437b73da3f75,3,3,1,2017-05-18 00:00:00,2017-05-18 00:00:00


In [0]:
duplicate_review_samples = (
    reviews_df
    .join(
        review_id_details.select("review_id").limit(5),
        on="review_id",
        how="inner"
    )
    .orderBy("review_id")
)

display(duplicate_review_samples)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
2172867fd5b1a55f98fe4608e1547b4b,e11ba7fd8fe0728dcd89efddcda9fb11,5,null,Entrega no prazo e produto de qualidade!,2018-02-15 00:00:00,2018-02-26 15:53:18,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
2172867fd5b1a55f98fe4608e1547b4b,559d606ac642899e44550f194fec7e08,5,null,Entrega no prazo e produto de qualidade!,2018-02-15 00:00:00,2018-02-26 15:53:18,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
2172867fd5b1a55f98fe4608e1547b4b,ac6e61336e852cdc45fe59ada3763a66,5,null,Entrega no prazo e produto de qualidade!,2018-02-15 00:00:00,2018-02-26 15:53:18,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
4d0e6dd087008d1f992d25ef6e1f619f,97f95535ada31d6fa853230e815ee4e3,4,null,null,2018-06-12 00:00:00,2018-06-17 23:48:04,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
4d0e6dd087008d1f992d25ef6e1f619f,cd0343141ac675cd40fab2c7fd2e99cc,4,null,null,2018-06-12 00:00:00,2018-06-17 23:48:04,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
4d0e6dd087008d1f992d25ef6e1f619f,d415e95816c2f3a2a34e06cd519157cc,4,null,null,2018-06-12 00:00:00,2018-06-17 23:48:04,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
832acec9bbf4efe65c3fb6423d8b4ed7,cf954e7391c1981cc9090267aec67ee9,4,null,null,2018-05-10 00:00:00,2018-05-11 10:53:21,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
832acec9bbf4efe65c3fb6423d8b4ed7,b9a528ec4ad891d009b3dc50588218f1,4,null,null,2018-05-10 00:00:00,2018-05-11 10:53:21,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
832acec9bbf4efe65c3fb6423d8b4ed7,66f943fd426ac8716ef6b6b97ace8bd8,4,null,null,2018-05-10 00:00:00,2018-05-11 10:53:21,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv
9e25d6e3025e9b9a0fc7f03588d33e2b,869997fbe01f39d184956b5c6bccfdbe,1,Razoavel,Peço um produto por um código e vem outro totalmente diferente.,2018-08-31 00:00:00,2018-09-03 09:33:00,2026-11-15T22:19:18.084Z,olist_order_reviews_dataset.csv


In [0]:
# Validate composite review key
review_key_validation = (
    reviews_df
    .groupBy("review_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate (review_id, order_id) groups:",
    review_key_validation.count()
)

print(
    "Unique review-order combinations:",
    reviews_df.select("review_id", "order_id").distinct().count()
)

Duplicate (review_id, order_id) groups: 0
Unique review-order combinations: 99224


In [0]:

from pyspark.sql import functions as F
from functools import reduce

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver_quarantine")

table_config = {
    "customers": {
        "keys": ["customer_id"],
        "types": {
            "customer_zip_code_prefix": "int"
        }
    },
    "geolocation": {
        "keys": [],
        "types": {
            "geolocation_zip_code_prefix": "int",
            "geolocation_lat": "double",
            "geolocation_lng": "double"
        }
    },
    "order_items": {
        "keys": ["order_id", "order_item_id"],
        "types": {
            "order_item_id": "int",
            "price": "double",
            "freight_value": "double"
        }
    },
    "payments": {
        "keys": ["order_id", "payment_sequential"],
        "types": {
            "payment_sequential": "int",
            "payment_installments": "int",
            "payment_value": "double"
        }
    },
    "reviews": {
        "keys": ["review_id", "order_id"],
        "types": {
            "review_score": "int",
            "review_creation_date": "timestamp",
            "review_answer_timestamp": "timestamp"
        }
    },
    "orders": {
        "keys": ["order_id"],
        "types": {
            "order_purchase_timestamp": "timestamp",
            "order_approved_at": "timestamp",
            "order_delivered_carrier_date": "timestamp",
            "order_delivered_customer_date": "timestamp",
            "order_estimated_delivery_date": "timestamp"
        }
    },
    "products": {
        "keys": ["product_id"],
        "types": {
            "product_name_lenght": "int",
            "product_description_lenght": "int",
            "product_photos_qty": "int",
            "product_weight_g": "double",
            "product_length_cm": "double",
            "product_height_cm": "double",
            "product_width_cm": "double"
        }
    },
    "sellers": {
        "keys": ["seller_id"],
        "types": {
            "seller_zip_code_prefix": "int"
        }
    },
    "category_translation": {
        "keys": ["product_category_name"],
        "types": {}
    }
}

metadata_columns = ["ingestion_timestamp", "source_file"]

audit_records = []


# --------------------------------------------------
# 2. REUSABLE SILVER TRANSFORMATION FUNCTION
# --------------------------------------------------

def transform_to_silver(table_name, config):

    source_table = f"workspace.bronze.{table_name}"
    target_table = f"workspace.silver.{table_name}"
    quarantine_table = f"workspace.silver_quarantine.{table_name}"

    print(f"\nProcessing: {table_name}")

    source_df = spark.table(source_table)

    input_count = source_df.count()

    # Separate business columns from metadata
    business_columns = [
        c for c in source_df.columns
        if c not in metadata_columns
    ]

    df = source_df

    # Standardize string columns
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column_name,
                F.when(
                    F.col(column_name).isNull(),
                    None
                ).otherwise(F.trim(F.col(column_name)))
            )

    # Cast configured columns to appropriate data types
    for column_name, target_type in config["types"].items():
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                F.col(column_name).cast(target_type)
            )

    # Remove exact duplicate business records
    df = df.dropDuplicates(business_columns)

    deduplicated_count = df.count()

    # Identify rows with missing business keys
    keys = config["keys"]

    if keys:
        invalid_condition = reduce(
            lambda a, b: a | b,
            [F.col(k).isNull() for k in keys]
        )

        quarantine_df = df.filter(invalid_condition)
        valid_df = df.filter(~invalid_condition)

    else:
        quarantine_df = df.limit(0)
        valid_df = df

    # Save quarantined records
    quarantine_count = quarantine_df.count()

    (
        quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(quarantine_table)
    )

    # Save cleaned Silver table
    output_count = valid_df.count()

    (
        valid_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    # Record audit metrics
    audit_records.append({
        "table_name": table_name,
        "input_count": input_count,
        "deduplicated_count": deduplicated_count,
        "output_count": output_count,
        "quarantine_count": quarantine_count,
        "status": "SUCCESS"
    })

    print(f"Input records: {input_count}")
    print(f"After deduplication: {deduplicated_count}")
    print(f"Silver records: {output_count}")
    print(f"Quarantined records: {quarantine_count}")


# --------------------------------------------------
# 3. EXECUTE PIPELINE FOR ALL NINE TABLES
# --------------------------------------------------

for table_name, config in table_config.items():
    transform_to_silver(table_name, config)


# --------------------------------------------------
# 4. CREATE DATA QUALITY AUDIT TABLE
# --------------------------------------------------

audit_df = spark.createDataFrame(audit_records)

audit_df = audit_df.withColumn(
    "pipeline_run_timestamp",
    F.current_timestamp()
)

(
    audit_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.data_quality_audit")
)

print("\n================================")
print("SILVER PIPELINE EXECUTION DONE")
print("================================")

display(audit_df)



Processing: customers
Input records: 99441
After deduplication: 99441
Silver records: 99441
Quarantined records: 0

Processing: geolocation
Input records: 1000163
After deduplication: 738332
Silver records: 738332
Quarantined records: 0

Processing: order_items
Input records: 112650
After deduplication: 112650
Silver records: 112650
Quarantined records: 0

Processing: payments
Input records: 103886
After deduplication: 103886
Silver records: 103886
Quarantined records: 0

Processing: reviews
Input records: 99224
After deduplication: 99224
Silver records: 99224
Quarantined records: 0

Processing: orders
Input records: 99441
After deduplication: 99441
Silver records: 99441
Quarantined records: 0

Processing: products
Input records: 32951
After deduplication: 32951
Silver records: 32951
Quarantined records: 0

Processing: sellers
Input records: 3095
After deduplication: 3095
Silver records: 3095
Quarantined records: 0

Processing: category_translation
Input records: 71
After deduplicatio

deduplicated_count,input_count,output_count,quarantine_count,status,table_name,pipeline_run_timestamp
99441,99441,99441,0,SUCCESS,customers,2026-11-15T22:24:39.482Z
738332,1000163,738332,0,SUCCESS,geolocation,2026-11-15T22:24:39.482Z
112650,112650,112650,0,SUCCESS,order_items,2026-11-15T22:24:39.482Z
103886,103886,103886,0,SUCCESS,payments,2026-11-15T22:24:39.482Z
99224,99224,99224,0,SUCCESS,reviews,2026-11-15T22:24:39.482Z
99441,99441,99441,0,SUCCESS,orders,2026-11-15T22:24:39.482Z
32951,32951,32951,0,SUCCESS,products,2026-11-15T22:24:39.482Z
3095,3095,3095,0,SUCCESS,sellers,2026-11-15T22:24:39.482Z
71,71,71,0,SUCCESS,category_translation,2026-11-15T22:24:39.482Z


In [0]:
# Verify all nine Silver tables

for table_name in table_config.keys():
    count = spark.table(
        f"workspace.silver.{table_name}"
    ).count()

    print(f"{table_name}: {count}")

# Review quality audit
display(
    spark.table("workspace.silver.data_quality_audit")
    .orderBy("table_name")
)

customers: 99441
geolocation: 738332
order_items: 112650
payments: 103886
reviews: 99224
orders: 99441
products: 32951
sellers: 3095
category_translation: 71


deduplicated_count,input_count,output_count,quarantine_count,status,table_name,pipeline_run_timestamp
71,71,71,0,SUCCESS,category_translation,2026-11-15T22:24:38.140Z
99441,99441,99441,0,SUCCESS,customers,2026-11-15T22:24:38.140Z
738332,1000163,738332,0,SUCCESS,geolocation,2026-11-15T22:24:38.140Z
112650,112650,112650,0,SUCCESS,order_items,2026-11-15T22:24:38.140Z
99441,99441,99441,0,SUCCESS,orders,2026-11-15T22:24:38.140Z
103886,103886,103886,0,SUCCESS,payments,2026-11-15T22:24:38.140Z
32951,32951,32951,0,SUCCESS,products,2026-11-15T22:24:38.140Z
99224,99224,99224,0,SUCCESS,reviews,2026-11-15T22:24:38.140Z
3095,3095,3095,0,SUCCESS,sellers,2026-11-15T22:24:38.140Z


In [0]:

from pyspark.sql import functions as F

# Load Silver tables
customers = spark.table("workspace.silver.customers")
orders = spark.table("workspace.silver.orders")
order_items = spark.table("workspace.silver.order_items")
products = spark.table("workspace.silver.products")
sellers = spark.table("workspace.silver.sellers")
payments = spark.table("workspace.silver.payments")
reviews = spark.table("workspace.silver.reviews")


# Define foreign-key relationships
relationships = [
    ("orders", orders, "customer_id", customers, "customer_id"),
    ("order_items", order_items, "order_id", orders, "order_id"),
    ("order_items", order_items, "product_id", products, "product_id"),
    ("order_items", order_items, "seller_id", sellers, "seller_id"),
    ("payments", payments, "order_id", orders, "order_id"),
    ("reviews", reviews, "order_id", orders, "order_id")
]


# Validate using left anti joins
fk_results = []

for child_name, child_df, child_key, parent_df, parent_key in relationships:

    child = child_df.alias("child")
    parent = parent_df.select(parent_key).distinct().alias("parent")

    orphan_count = (
        child.join(
            parent,
            F.col(f"child.{child_key}") == F.col(f"parent.{parent_key}"),
            "left_anti"
        )
        .filter(F.col(f"child.{child_key}").isNotNull())
        .count()
    )

    fk_results.append((
        child_name,
        child_key,
        orphan_count,
        "PASS" if orphan_count == 0 else "CHECK"
    ))

fk_audit = spark.createDataFrame(
    fk_results,
    ["table_name", "foreign_key", "orphan_records", "validation_status"]
)

display(fk_audit)


table_name,foreign_key,orphan_records,validation_status
orders,customer_id,0,PASS
order_items,order_id,0,PASS
order_items,product_id,0,PASS
order_items,seller_id,0,PASS
payments,order_id,0,PASS
reviews,order_id,0,PASS
